<a href="https://colab.research.google.com/github/gusti-amber/rag/blob/main/chapter5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 5.1 RunnableとRunnableSequence─LCELの最も基本的な構成要素

LCELの最も基本的な実装は、Prompt template・Chat model・Output parserの3つを連結することです。

前章で解説したように、Prompt template、Chat model、Output parserはすべて「invoke」メソッドで実行することができます。

LCELをよく理解できるよう、まずはこれらを順にinvokeすることを試してみます。

まず、prompt（ChatPromptTemplate）・model（ChatOpenAI）・output_parser（StrOutputParser）を準備します。

In [2]:
!pip install --upgrade openai langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.26.0
    Uninstalling openai-2.26.0:
      Successfully uninstalled openai-2.26.0


In [11]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "agent-book"

In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages(
  [
      ("system", "ユーザーが入力した料理のレシピを考えてください。"),
      ("human", "{dish}"),
  ]
)

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

output_parser = StrOutputParser()

In [6]:
prompt_value = prompt.invoke({"dish": "カレー"})
ai_message = model.invoke(prompt_value)
output = output_parser.invoke(ai_message)

print(output)

カレーのレシピをご紹介します。シンプルで美味しい基本のカレーを作りましょう。

### 材料（4人分）
- 鶏肉（もも肉または胸肉）: 400g
- 玉ねぎ: 2個
- にんじん: 1本
- じゃがいも: 2個
- カレールー: 1箱（約200g）
- サラダ油: 大さじ2
- 水: 800ml
- 塩: 適量
- 胡椒: 適量
- お好みでガーリックパウダーや生姜: 適量

### 作り方
1. **材料の下ごしらえ**:
   - 鶏肉は一口大に切り、塩と胡椒をふっておきます。
   - 玉ねぎは薄切り、にんじんは輪切り、じゃがいもは一口大に切ります。

2. **炒める**:
   - 大きめの鍋にサラダ油を熱し、玉ねぎを中火で炒めます。玉ねぎが透明になるまで炒めます。
   - 鶏肉を加え、表面が白くなるまで炒めます。

3. **野菜を加える**:
   - にんじんとじゃがいもを鍋に加え、全体をよく混ぜます。

4. **煮る**:
   - 水を加え、強火で煮立たせます。煮立ったら、アクを取り除き、中火にして蓋をし、約15分煮ます。

5. **カレールーを加える**:
   - カレールーを割り入れ、よく溶かします。さらに10分ほど煮込み、全体がなじんだら味を見て、必要に応じて塩や胡椒で調整します。

6. **仕上げ**:
   - お好みでガーリックパウダーや生姜を加えて風味をアップさせます。火を止めて、少し冷ますと味がなじみます。

7. **盛り付け**:
   - ご飯と一緒に盛り付けて、お好みで福神漬けやらっきょうを添えて完成です。

### おすすめのトッピング
- 煮卵
- チーズ
- パクチー

この基本のカレーはアレンジがしやすいので、野菜や肉を変えて楽しんでください！


このようなコードでもPrompt template、Chat model、Output parserを順に実行できますが、LCELではこれらを「|」で連結してから実行します。

実は、ChatPromptTemplate・ChatOpenAI・StrOutputParserは、すべてLangChainの「Runnable」という抽象基底クラスを継承しています。
Runnableを「|」でつなぐと「RunnableSequence」になります。
RunnableSequenceもRunnableの一種です。

In [ ]:
chain = prompt | model | output_parser
output = chain.invoke({"dish": "カレー"})

このように、Runnableを「|」でつないで新たなRunnableを作り、それをinvokeしたときに内部のRunnableが順にinvokeされるというのがLCELの基本です。

### Runnableの実行方法─invoke・stream・batch

### LCELの「|」でさまざまなRunnableを連鎖させる

In [12]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

output_parser = StrOutputParser()

In [13]:
cot_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "ユーザーの質問にステップバイステップで回答してください。"),
        ("human", "{question}"),
    ]
)

cot_chain = cot_prompt | model | output_parser

In [14]:
summarize_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "ステップバイステップで考えた回答から結論だけ抽出してください"),
        ("human", "{text}"),
    ]
)
summarize_chain = summarize_prompt | model | output_parser

In [15]:
cot_summarize_chain = cot_chain | summarize_chain
cot_summarize_chain.invoke({"question": "10 + 2 * 3"})

'10 + 2 * 3 の答えは **16** です。'

最終的に要約されたシンプルな回答が得られました。このときcot_summarize_chainの内部では、まずcot_chainが実行され、ステップバイステップで考えた冗長な回答を得ます。その回答を入力としてsummarize_chainを実行し、要約されたシンプルな回答を得ています。LLMを2回呼び出すことで、Zero-shot CoTを使って回答の精度を高めつつ、最終的にはシンプルな出力を得ることができたということです。